In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import os, json

In [2]:
TYPE_DIR = os.path.join('skin_dataset', 'Skintype')

classes = [d for d in os.listdir(TYPE_DIR) if os.path.isdir(os.path.join(TYPE_DIR, d))]

for cls in classes:
    path = os.path.join(TYPE_DIR, cls)
    num_images = len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print(f"{cls}: {num_images} images")


combination: 30 images
dry: 30 images
normal: 30 images
oily: 30 images


In [4]:
# %% settings
TYPE_DIR = os.path.join('skin_dataset','Skintype')
IMAGE_SIZE = (224,224)
BATCH_SIZE = 32
EPOCHS = 30
MODEL_OUT = 'model_skintype.h5'


# %% data
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2,
rotation_range=15, horizontal_flip=True)
train_gen = train_datagen.flow_from_directory(
TYPE_DIR,
target_size=IMAGE_SIZE,
batch_size=BATCH_SIZE,
class_mode='categorical',
subset='training'
)
val_gen = train_datagen.flow_from_directory(
TYPE_DIR,
target_size=IMAGE_SIZE,
batch_size=BATCH_SIZE,
class_mode='categorical',
subset='validation'
)


# %% model
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE,3))
base.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(train_gen.num_classes, activation='softmax')(x)
model = models.Model(inputs=base.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# %% train
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)


# %% save
model.save(MODEL_OUT)
with open('label_map_skintype.json','w') as f:
    json.dump(train_gen.class_indices, f)
print('Saved', MODEL_OUT)

Found 96 images belonging to 4 classes.
Found 24 images belonging to 4 classes.
Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 21s 8s/step - accuracy: 0.3490 - loss: 1.6988 - val_accuracy: 0.3750 - val_loss: 1.4606
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 6s/step - accuracy: 0.5794 - loss: 0.9550 - val_accuracy: 0.5000 - val_loss: 1.4750
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 6s/step - accuracy: 0.7591 - loss: 0.6619 - val_accuracy: 0.4167 - val_loss: 1.5585
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 7s/step - accuracy: 0.7969 - loss: 0.4591 - val_accuracy: 0.5000 - val_loss: 1.4945
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 19s 6s/step - accuracy: 0.8620 - loss: 0.3630 - val_accuracy: 0.4583 - val_loss: 1.5755
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 17s 6s/step - accuracy: 0.9089 - loss: 0.2884 - val_accuracy: 0.4583 - val_loss: 1.8132
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 17s 6s/step - accuracy: 0.9310 - loss: 0.2333 - val_accuracy: 0.4167 - val_loss: 1.9327
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - ac

Saved model_skintype.h5
